# 09 — Refine and validate Earth candidates

This notebook implements stages 1 and 2 without repeating the global search. It refines the top cells from notebook 08 on a 1-degree local grid, applies a real coastline/land mask, re-queries NASA POWER, and inventories MODIS follow-up coverage through NASA CMR.

This is a validation shortlist, not proof of active dust transport.


In [ ]:
from pathlib import Path
import json, math
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, requests
import matplotlib.pyplot as plt
from IPython.display import display

OUT = Path('/kaggle/working/earth_refined_validation'); OUT.mkdir(parents=True, exist_ok=True)
ROOTS = [Path('/kaggle/input/nasa-hackathon-08-earth-matching'), Path('/kaggle/working/earth_dust_matching'), Path('/kaggle/input')]
def find_file(name):
    for root in ROOTS:
        if root.exists():
            hits = list(root.rglob(name))
            if hits: return hits[0]
    raise FileNotFoundError(name)

matches = pd.read_parquet(find_file('earth_dust_environment_matches.parquet'))
top = matches.sort_values('rank').head(10).copy()
print('Global seed cells:', len(top))
display(top[['rank','latitude','longitude','analog_score']])


## 1. Denser local grid and real land mask

Each seed cell receives a local 7×7-degree window at 1-degree spacing. Natural Earth land polygons remove ocean cells. Elevation is retained as a measurement, not used as the land mask.


In [ ]:
LAND_URL = 'https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_110m_land.geojson'
land_path = OUT/'ne_110m_land.geojson'
if not land_path.exists():
    r = requests.get(LAND_URL, timeout=60); r.raise_for_status(); land_path.write_bytes(r.content)
land_geo = json.loads(land_path.read_text())
polygons = []
def collect(g):
    if g['type'] == 'Polygon': polygons.append(g['coordinates'])
    elif g['type'] == 'MultiPolygon': polygons.extend(g['coordinates'])
for f in land_geo['features']: collect(f['geometry'])
def ring_inside(x, y, ring):
    hit = False
    for (x1,y1),(x2,y2) in zip(ring, ring[1:]+ring[:1]):
        if ((y1 > y) != (y2 > y)) and x < (x2-x1)*(y-y1)/(y2-y1+1e-15)+x1: hit = not hit
    return hit
def on_land(lon, lat):
    return any(ring_inside(lon,lat,p[0]) and not any(ring_inside(lon,lat,h) for h in p[1:]) for p in polygons)

pts = []
for r in top.itertuples():
    for lat in np.arange(max(-89,float(r.latitude)-3), min(89,float(r.latitude)+3.01), 1.0):
        for lon in np.arange(float(r.longitude)-3, float(r.longitude)+3.01, 1.0):
            lon = ((lon+180)%360)-180
            if on_land(lon, lat): pts.append((round(lat,4),round(lon,4),int(r.rank)))
grid = pd.DataFrame(pts, columns=['latitude','longitude','parent_global_rank']).drop_duplicates(['latitude','longitude']).reset_index(drop=True)
print('Refined land cells:', len(grid))


## 2. Re-query NASA POWER and rescore

The same documented POWER climatology variables as notebook 08 are used: albedo, precipitation, wind, surface temperature, monthly temperature range, and elevation. Scores are recomputed against the Mars source distribution; no new process label is invented.


In [ ]:
POWER = 'https://power.larc.nasa.gov/api/temporal/climatology/point'
def fetch(row):
    q = {'parameters':'TS,PRECTOTCORR,WS10M,ALLSKY_SRF_ALB','community':'AG','longitude':float(row.longitude),'latitude':float(row.latitude),'start':'2001','end':'2025','format':'JSON'}
    try:
        p = requests.get(POWER, params=q, timeout=45).json(); d = p['properties']['parameter']; vals = [float(v) for v in d['TS'].values() if float(v)>-900]
        return {**row.to_dict(),'earth_surface_albedo_ann':float(d['ALLSKY_SRF_ALB']['ANN']),'earth_precipitation_mm_day_ann':float(d['PRECTOTCORR']['ANN']),'earth_wind_mps_ann':float(d['WS10M']['ANN']),'earth_surface_temperature_c_ann':float(d['TS']['ANN']),'earth_surface_temperature_range_c':max(vals)-min(vals),'elevation_m':float(p['geometry']['coordinates'][2]),'power_status':'success'}
    except Exception as e: return {**row.to_dict(),'power_status':type(e).__name__}

with ThreadPoolExecutor(max_workers=16) as ex:
    refined = pd.DataFrame(list(ex.map(fetch, [r for _,r in grid.iterrows()])))
refined = refined[refined.power_status == 'success'].copy()
refined['earth_annual_precipitation_mm'] = refined.earth_precipitation_mm_day_ann*365.25
refined['earth_aridity_score'] = -np.log1p(refined.earth_annual_precipitation_mm.clip(lower=0))
refined['earth_dry_windy_proxy'] = (refined.earth_aridity_score.rank(pct=True)+refined.earth_wind_mps_ann.rank(pct=True))/2
mars = pd.read_parquet(find_file('mars_candidate_validation_results.parquet'))
mars = mars[mars['validated_interpretation'] == 'fine_particulate_dust_compatible_surface'].copy()
mf = ['raw_mean__albedo_mean','raw_mean__thermal_inertia_mean','raw_mean__quality_masked__mineral_surface_dust']
ef = ['earth_surface_albedo_ann','earth_surface_temperature_range_c','earth_dry_windy_proxy']
mr = mars[mf].rank(pct=True).to_numpy(); er = refined[ef].rank(pct=True).to_numpy()
refined['marginal_similarity'] = 1-np.mean(np.abs(er-mr.mean(axis=0)),axis=1)
refined['joint_similarity'] = 1-np.min(np.linalg.norm(er[:,None,:]-mr[None,:,:],axis=2),axis=1)/math.sqrt(3)
refined['preliminary_analog_score'] = (refined.marginal_similarity+refined.joint_similarity)/2
coords = refined[['latitude','longitude']].to_numpy()
refined['spatial_coherence'] = [float(np.mean(np.sqrt((coords[:,0]-a)**2+(coords[:,1]-b)**2)<=1.5)) for a,b in coords]
refined['analog_score'] = .8*refined.preliminary_analog_score + .2*refined.spatial_coherence
refined = refined.sort_values('analog_score',ascending=False).reset_index(drop=True); refined['rank'] = np.arange(1,len(refined)+1)
display(refined.head(20))
refined.to_parquet(OUT/'refined_earth_matches.parquet', index=False)


## 3. MODIS follow-up inventory through NASA CMR

CMR metadata identifies higher-resolution observations to inspect next. It is not used as a measurement in the score.


In [ ]:
CMR = 'https://cmr.earthdata.nasa.gov/search/granules.json'
collections = {'MODIS_aerosol':'MOD04_L2', 'MODIS_surface_reflectance':'MOD09GA'}
def inventory(row):
    out = {**row[['rank','latitude','longitude','analog_score']].to_dict()}
    for name, short in collections.items():
        q = {'short_name':short,'page_size':10,'downloadable':'true','bounding_box':f'{row.longitude-1},{row.latitude-1},{row.longitude+1},{row.latitude+1}'}
        try:
            j = requests.get(CMR, params=q, timeout=45).json(); out[name+'_granules'] = len(j.get('feed',{}).get('entry',[])); out[name+'_status'] = 'success'
        except Exception as e: out[name+'_granules'] = np.nan; out[name+'_status'] = type(e).__name__
    return out
with ThreadPoolExecutor(max_workers=8) as ex:
    inventory_df = pd.DataFrame(list(ex.map(inventory, [r for _,r in refined.head(20).iterrows()])))
inventory_df.to_parquet(OUT/'modis_followup_inventory.parquet', index=False)
display(inventory_df.head(10))
summary = {'notebook':'09_earth_refined_validation','global_seed_cells':int(len(top)),'refined_land_cells':int(len(refined)),'modis_followup_cells':int(len(inventory_df)),'top_10':refined.head(10)[['rank','latitude','longitude','analog_score']].to_dict('records'),'interpretation':'higher-resolution environmental validation shortlist; not proof of active aeolian transport'}
(OUT/'validation_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(json.dumps(summary, indent=2))
